# 07 - model testing (sample)

final inference pipeline: load model, run on held-out test sample, export results.

In [ ]:
import joblib, pandas as pd, numpy as np, librosa
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from pathlib import Path
from src.config.settings import SAMPLE_RATE, N_MFCC, N_FFT, HOP_LENGTH

SAMPLES = Path("../../data/samples")
df = pd.read_csv(SAMPLES / "sample_labels.csv")
svm = joblib.load(SAMPLES / "svm_baseline_sample.pkl")

In [ ]:
def extract_mfcc(fp):
    y,sr = librosa.load(fp, sr=SAMPLE_RATE)
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    d = librosa.feature.delta(m)
    d2 = librosa.feature.delta(m, order=2)
    return np.concatenate([m.mean(1), m.std(1), d.mean(1), d.std(1), d2.mean(1), d2.std(1)])

In [ ]:
test = df[df['split']=='test']
print(f"testing on {len(test)} files (actors {sorted(test['actor'].unique())})")
X_test = np.array([extract_mfcc(fp) for fp in test['filepath']])
y_test = test['emotion_code'].values
scaler = StandardScaler().fit(X_test)
y_pred = svm.predict(scaler.transform(X_test))
acc = accuracy_score(y_test, y_pred)
print(f"test accuracy: {acc*100:.1f}%")

In [ ]:
results = test.copy()
results['predicted'] = y_pred
results['correct'] = (y_test == y_pred).astype(int)
csv_path = SAMPLES / 'predictions_sample.csv'
results.to_csv(csv_path, index=False)
print(f"predictions saved: {csv_path}")
print(f"per-speaker accuracy:")
for actor in sorted(test['actor'].unique()):
    sub = results[results['actor']==actor]
    a = sub['correct'].mean() * 100
    print(f"  Actor_{actor:02d}: {a:.1f}% ({sub['correct'].sum()}/{len(sub)})")

testing complete. predictions exported for midterm report analysis.